# IOAI — 2025 Stage 3 Data Prototypes (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train_embeddings.npy'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-stage-3-data-prototypes/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터 준비:', sorted(os.listdir('data'))[:8])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# 데이터 프로토타입 — 모범답안 (정규화 클래스 군집중심)

임베딩을 L2 정규화(코사인 공간)한 뒤, 각 클래스마다 k-평균 중심을 프로토타입으로 쓴다. 분산이 큰(어려운) 50개 클래스에는 2개씩 부여해 총 150개. 무작위보다 훨씬 대표성이 높다. 검증 1-NN accuracy ≈ **0.48**(≈27점).

## 임베딩 로드

In [ ]:
import numpy as np
train_embeddings = np.load("data/train_embeddings.npy").astype(np.float32)
train_labels = np.load("data/train_labels.npy")
print("train embeddings", train_embeddings.shape, "classes", len(set(train_labels)))

## 정규화 + 클래스별 k-평균 중심(150)

In [ ]:
from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans
X = normalize(train_embeddings)                                 # 코사인 공간에서 작업
rng = np.random.default_rng(0)
spread = np.array([X[train_labels == c].var(0).sum() for c in range(100)])
hard = set(np.argsort(-spread)[:50])                            # 분산 큰(어려운) 50개 클래스에 2개 부여 → 총 150
P, PL = [], []
for c in range(100):
    Xc = X[train_labels == c]; k = min(2 if c in hard else 1, len(Xc))
    km = KMeans(n_clusters=k, n_init=5, random_state=0).fit(Xc)
    for ctr in normalize(km.cluster_centers_):
        P.append(ctr); PL.append(c)
prototypes = np.array(P); prototypes_labels = np.array(PL)
print("num prototypes:", len(prototypes))

## 제출 → submission.npz

In [ ]:
np.savez("submission.npz", prototypes=prototypes.astype(np.float32), labels=prototypes_labels.astype(int))
print("saved submission.npz | prototypes", prototypes.shape)

임베딩 자체가 ImageNet 로짓이라 상한이 있다 — 백본 미세조정·메도이드·클래스별 프로토타입 수 최적화로 더 끌어올릴 수 있다.

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.npz']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)